# 1. Data Preprocessing: Raw EndomondoHR JSON to CSV

**Purpose.** First step of the RunSight training pipeline. The raw EndomondoHR (FitRec) file is a 4.6 GB text file with one workout record per line, written in Python-dict syntax rather than strict JSON. This notebook parses it line by line in memory-safe batches, keeps only running workouts (`sport == 'run'`), and saves the result as a CSV for the next notebooks.

**Inputs**
- `data/endomondoHR_proper.json` - raw FitRec dataset (download link in the first cell)

**Outputs**
- `artifacts/batches/batch_*.pkl` and `artifacts/batches/state.txt` - intermediate batches and the resume checkpoint
- `data/endomondoHR_proper.csv` - all running workouts as one table

Part of RunSight - see the repo README.

In [ ]:
# print("downloading file...")
# !mkdir -p data
# !wget -O data/endomondoHR_proper.json "https://mcauleylab.ucsd.edu/public_datasets/gdrive/fitrec/endomondoHR_proper.json"

## Data source

This notebook is the first step in turning the raw JSON file of the
EndomondoHR (FitRec) dataset into a CSV that is practical to work with.
The file is 4.6 GB. It is stored at `data/endomondoHR_proper.json`, so it
does not have to be downloaded again in every session.

In [ ]:
import json
import ast, os, pickle, glob
import pandas as pd

In [ ]:
path = "data/endomondoHR_proper.json"

## The file format is not what we expected

This is not a standard JSON file (one big `[...]` array). Instead, every
LINE is a separate workout record (similar to JSON Lines / NDJSON).
Reading a text file line by line in Python always yields each line as a
`str`; that is normal Python behavior, not a problem with the file, and
the cell below only confirms it.

The real problem: the standard `json.loads()` raised an error on every
line. The reason is that the lines are stored in **Python dict syntax, not
real JSON**. The JSON standard requires double quotes around strings
(`"key"`), but the lines in this file use single quotes (`'key'`), which
is valid in Python but invalid in JSON. `json.loads` treats these
single-quoted lines as "malformed JSON" and rejects them.

In [ ]:
n=0
with open(path) as f:
  for line in f:
    n+=1
    print(type(line))
    if n==10:
      break

## First attempt: manual string splitting was not enough

Since `json.loads` did not work, the first simple idea was to split each
line on the single-quote character (`'`) and stitch the pieces back
together by hand (`str.split("'")`). This approach is fragile: text
fields in the data (e.g. `url`, `gender`) can also contain single quotes,
there are nested lists and dicts, and numbers and strings are mixed.
Manually cutting the string apart and reassembling it in the right order
is error-prone and not maintainable. The output of the cell below
confirms this: instead of a meaningful structure, the line is split into
scattered text fragments.

In [ ]:
n = 0
with open(path) as f:
    for line in f:
        n += 1
        d = line.split(sep="'")
        k=0
        for i in d:
          print(k, d[k])
          k+=1
        if n == 1:
            break

## Solution: `ast.literal_eval`

The lines are actually valid **Python** dict/list syntax (single-quoted
strings, Python `dict`/`list` syntax); they are just not valid JSON.
`literal_eval` from the `ast` (Abstract Syntax Tree) library exists for
exactly this case: unlike `eval()`, it does not execute arbitrary code
(no security risk). It only parses constant Python expressions (numbers,
strings, lists, dicts, tuples) and turns the text into a real Python
object. Since the lines were saved in Python dict syntax, using
`ast.literal_eval` instead of `json.loads` solved the problem. In the
cell below, the `sport` field is read successfully from every line.

In [ ]:
with open (path,'r') as f:
  for line in f:
    w = ast.literal_eval(line)
    print(w.get('sport'))

## Extract only the running workouts

The raw dataset is mixed: besides running it contains many other sports
such as walking, cycling and swimming. Since our project focuses on
running (aerobic/anaerobic run classification), we check the `sport`
field of each line and keep only the `"run"` records, skipping the rest.
We do this filtering while READING the file line by line, without loading
all the data into memory. The next section explains why this is
necessary (memory/RAM limit).

In [ ]:
def extract_runs(path, out_dir, batch=20000):
    os.makedirs(out_dir, exist_ok=True)
    state = f'{out_dir}/state.txt'
    start = int(open(state).read()) if os.path.exists(state) else 0
    n_batch = len(glob.glob(f'{out_dir}/batch_*.pkl'))
    buf = []
    with open(path) as f:
        for i, line in enumerate(f):
            if i < start:
                continue
            try:
                w = ast.literal_eval(line)
                if w.get('sport') == 'run':
                    buf.append(w)
            except Exception:
                continue
            if (i + 1) % batch == 0:
                pickle.dump(buf, open(f'{out_dir}/batch_{n_batch}.pkl', 'wb'))
                n_batch += 1
                buf.clear()
                open(state, 'w').write(str(i + 1))
                print(f'{i+1} lines processed')
    if buf:
        pickle.dump(buf, open(f'{out_dir}/batch_{n_batch}.pkl', 'wb'))
    open(state, 'w').write(str(i + 1))

def load_runs_df(out_dir):
    dfs = [pd.DataFrame(pickle.load(open(p, 'rb')))
           for p in sorted(glob.glob(f'{out_dir}/batch_*.pkl'))]
    return pd.concat(dfs, ignore_index=True)

## RAM limit: why we process in batches

Trying to read the whole 4.6 GB JSON file, convert every line into a
Python object and accumulate everything in a single list/DataFrame filled
the limited RAM that Colab provides; the session crashed and restarted on
its own (all progress up to that point was lost). The solution: read the
file line by line and, every time a given number of lines (`batch=20000`)
has accumulated, save it to disk as a separate file (`.pkl`) and clear it
from memory. This way only the latest batch is held in memory AT ANY TIME,
not the whole dataset.

A `state.txt` file also records the last line that was processed. If the
session crashes again, processing can resume from that line instead of
starting over. This is a standard resilience technique for large,
long-running data jobs: split the work into small, repeatable pieces and
save progress.

In [ ]:
os.makedirs('artifacts', exist_ok=True)
extract_runs('data/endomondoHR_proper.json',
             'artifacts/batches')
df = load_runs_df('artifacts/batches')
print(df.shape)

## First observation: speed data is mostly missing

Only about 11,500 of the 70,591 filtered running workouts have directly
recorded speed data (see the "non-null" count of the `speed` column in
the `df.info()` output); in the remaining ~60,000 rows `speed` is empty
(`NaN`). This is not a blocker for the project, because speed can be
derived from the position data (latitude/longitude together with
timestamps): compute the distance between two consecutive GPS points and
divide it by the elapsed time.

However, this derived speed will be NOISY in its raw form. GPS
measurements are not perfect, and problems such as signal jumps and
repeated timestamps cause meaningless spikes in the computed speed.
Cleaning this raw speed and making it usable is handled in a separate
notebook (`3_haversine_savgol_filter.ipynb`). The job of this notebook
ends here: converting the data from raw JSON into a workable CSV.

In [ ]:
df.info()

In [ ]:
df.head()

## Save the intermediate result

Colab sessions can time out or close unexpectedly (it already happened
once above because of RAM). In that case the in-memory `df` is lost and
the whole process (download, parsing, filtering, batching) would have to
be repeated. To prevent that, we save the clean DataFrame we will work
with as a persistent CSV file. Later notebooks (e.g.
`3_haversine_savgol_filter.ipynb`) can then read this CSV directly
without repeating these heavy preprocessing steps.

In [ ]:
os.makedirs('data', exist_ok=True)
df_to_csv = df.to_csv('data/endomondoHR_proper.csv')